# 标准 Transformer 架构全解析企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 NLP / 深度学习算法岗面试手撕  
> **核心涵盖**：正余弦位置编码、缩放点积注意力、因果与填充双掩码融合、多头自注意力 (MHA)、交叉注意力 (Cross-Attention)、FFN、Pre-LN/Post-LN 残差网络、Encoder/Decoder 模块组装、自回归推理循环  
> **设计准则**：纯 PyTorch 逐行白盒手写，剔除 `nn.Transformer` 黑盒封装，吃透多头张量变换与数值稳定性。

---
### 核心模块速览
1. **模块一**：正弦/余弦绝对位置编码 (Sinusoidal Positional Encoding) 矩阵向量化推导
2. **模块二**：缩放点积注意力 (Scaled Dot-Product Attention) 与 $\sqrt{d_k}$ 缩放因子
3. **模块三**：因果掩码 (Causal Mask) 与 Padding 掩码双向融合
4. **模块四**：标准多头自注意力机制 (Multi-Head Attention, MHA) 纯手撕
5. **模块五**：交叉注意力机制 (Cross-Attention / Encoder-Decoder Attention) 手撕
6. **模块六**：前馈神经网络 (Position-wise Feed-Forward Network, FFN)
7. **模块七**：Pre-LN vs. Post-LN 架构差异与残差连接
8. **模块八**：标准 Transformer Encoder Block 与 Decoder Block 端到端组合手撕
9. **模块九**：端到端自回归翻译/生成推理解码循环

---
## 模块一：正弦/余弦绝对位置编码 (Sinusoidal Positional Encoding) 矩阵向量化推导

### 【笔试考点与公式】
- **公式**：
  $$PE_{(pos, 2i)} = \sin\left( \frac{pos}{10000^{\frac{2i}{d}}} \right), \quad PE_{(pos, 2i+1)} = \cos\left( \frac{pos}{10000^{\frac{2i}{d}}} \right)$$
- **广播技巧**：令 $div\_term = \exp\left( -\frac{2i}{d} \ln 10000 \right)$，利用外积 $pos \otimes div\_term$ 一行完成整个矩阵的计算，严禁写多层循环！

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # pe 形状: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)
        
        # 利用 exp(log) 技巧计算分母
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 注册为 buffer，不参与梯度反传，随模型保存
        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# 测试验证位置编码
pos_enc = SinusoidalPositionalEncoding(d_model=64, max_len=100)
dummy_x = torch.zeros(2, 10, 64)
x_with_pe = pos_enc(dummy_x)
print("位置编码注入后张量形状:", x_with_pe.shape)
assert x_with_pe.shape == (2, 10, 64)
# 验证首个 token 的 sin(0)=0, cos(0)=1
print("第 0 位置前两个维度值 (sin=0, cos=1):", x_with_pe[0, 0, :2].numpy())
assert torch.isclose(x_with_pe[0, 0, 0], torch.tensor(0.0))
assert torch.isclose(x_with_pe[0, 0, 1], torch.tensor(1.0))
print(">>> 正弦余弦位置编码验证通过！")

位置编码注入后张量形状: torch.Size([2, 10, 64])
第 0 位置前两个维度值 (sin=0, cos=1): [0. 1.]
>>> 正弦余弦位置编码验证通过！


In [ ]:
# 练习

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Sin_Cos_Pos_embedding(nn.Module):
    def __init__(self,d_model,max_len = 100):
        super(Sin_Cos_Pos_embedding, self).__init__()
        pe = torch.zeros(max_len,d_model)
        mul_item = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(torch.arange(0,max_len).unsqueeze(1).float()*mul_item)
        pe[:,1::2] = torch.cos(torch.arange(0,max_len).unsqueeze(1).float()*mul_item)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x):
        seq_len = x.size(1)
        return x + self.pe[:,:seq_len,:]    
#测试
pos_embed = Sin_Cos_Pos_embedding(d_model=512,max_len=100)
x = torch.randn(1, 10, 512)
output = pos_embed(x)
print("位置编码注入后张量形状:", output.shape)

位置编码注入后张量形状: torch.Size([1, 10, 512])


---
## 模块二：缩放点积注意力 (Scaled Dot-Product Attention) 与 $\sqrt{d_k}$ 缩放因子

### 【笔试考点与面试必问】
1. **公式**：
   $$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} + M \right) V$$
2. **为什么除以 $\sqrt{d_k}$？**：
   若 $q_i, k_i$ 是均值为 0、方差为 1 的独立随机变量，点积 $\sum_{i=1}^{d_k} q_i k_i$ 的方差为 $d_k$。当 $d_k$ 较大时，点积数值极大，推入 Softmax 会进入**极度平坦的饱和区，梯度几近为 0**！除以 $\sqrt{d_k}$ 使方差恢复为 1。

In [3]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (..., seq_len, d_k)
    mask: (..., seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # 将 mask 为 0 的位置填入 -1e9 极小值
        scores = scores.masked_fill(mask == 0, -1e9)
        
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# 测试注意力与缩放
dummy_q = torch.randn(2, 4, 32)
dummy_k = torch.randn(2, 4, 32)
dummy_v = torch.randn(2, 4, 32)
out_attn, weights = scaled_dot_product_attention(dummy_q, dummy_k, dummy_v)
print("缩放点积注意力输出形状:", out_attn.shape)
assert out_attn.shape == (2, 4, 32)
assert torch.allclose(weights.sum(dim=-1), torch.ones(2, 4))
print(">>> 缩放点积注意力验证成功！")

缩放点积注意力输出形状: torch.Size([2, 4, 32])
>>> 缩放点积注意力验证成功！


---
## 模块三：因果掩码 (Causal Mask) 与 Padding 掩码双向融合

### 【笔试考点】
1. **因果掩码 (下三角矩阵)**：`torch.tril(torch.ones(L, L))`，防止解码时偷看未来 Token；
2. **填充掩码 (Padding Mask)**：排除 `<PAD>` 占位符对注意力的干扰；
3. **双掩码融合**：取两者逻辑与 `mask = causal_mask & pad_mask`。

In [ ]:
def create_masks(src, tgt, pad_idx=0):
    """
    构建 Encoder Padding Mask 与 Decoder Causal + Pad 复合掩码
    src: (B, src_len)
    tgt: (B, tgt_len)
    """
    # 1. src padding mask: (B, 1, 1, src_len)
    src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
    
    # 2. tgt padding mask: (B, 1, tgt_len, 1)
    tgt_pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
    
    # 3. tgt causal mask (下三角): (1, 1, tgt_len, tgt_len)
    tgt_len = tgt.size(1)
    causal_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
    
    # 融合 Decoder 复合掩码
    tgt_mask = tgt_pad_mask & causal_mask
    return src_mask, tgt_mask

# 测试掩码生成
dummy_src = torch.tensor([[1, 2, 3, 0]]) # 最后一个是 PAD
dummy_tgt = torch.tensor([[5, 6, 7]])
s_mask, t_mask = create_masks(dummy_src, dummy_tgt, pad_idx=0)
print("源端 Mask 形状与内容 (PAD 为 False):", s_mask.shape, s_mask[0, 0, 0].numpy())
print("目标端因果 Mask (3x3 下三角):\n", t_mask[0, 0].numpy())
assert t_mask[0, 0, 0, 1] == False and t_mask[0, 0, 1, 0] == True
print(">>> 因果掩码与填充掩码融合验证成功！")

---
## 模块四：标准多头自注意力机制 (Multi-Head Attention, MHA) 纯手撕

### 【笔试考点与张量变换】
- **核心维度流**：
  $$(B, L, D) \xrightarrow{W_q} (B, L, D) \xrightarrow{\text{view}} (B, L, H, D/H) \xrightarrow{\text{transpose(1, 2)}} (B, H, L, D/H)$$
- 将多头合并回输出：
  $$(B, H, L, D/H) \xrightarrow{\text{transpose(1, 2)}} (B, L, H, D/H) \xrightarrow{\text{contiguous().view}} (B, L, D) \xrightarrow{W_o} (B, L, D)$$
- **雷区**：在 `view` 之前必须调用 **`.contiguous()`**，否则张量内存不连续会报错！

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        
        # 1. 线性投影并拆分多头: (B, num_heads, L, d_k)
        Q = self.W_q(q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 批量多头注意力计算
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        
        # 3. 拼接多头并线性变换还原维度
        # (B, H, L, d_k) -> (B, L, H, d_k) -> (B, L, d_model)
        concat = attn_out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        output = self.W_o(concat)
        return output

# 测试 MHA
mha = MultiHeadAttention(d_model=64, num_heads=8)
x_in = torch.randn(2, 6, 64)
mha_out = mha(x_in, x_in, x_in)
print("MHA 输出形状:", mha_out.shape)
assert mha_out.shape == (2, 6, 64)
print(">>> Multi-Head Attention 验证成功！")

---
## 模块五：交叉注意力机制 (Cross-Attention / Encoder-Decoder Attention)

### 【笔试考点与本质】
- **自注意力 vs 交叉注意力**：
  - 自注意力：$Q, K, V$ 均来自同一个序列；
  - 交叉注意力：$Q$ 来自解码器上一层输出，$K, V$ 来自编码器的最终记忆输出 `memory`！
  - 掩码使用 `src_mask`（屏蔽源端的 Pad）。

In [ ]:
def cross_attention_demo():
    cross_attn = MultiHeadAttention(d_model=64, num_heads=8)
    dec_state = torch.randn(2, 4, 64)   # 解码端序列长 4
    enc_memory = torch.randn(2, 10, 64) # 编码端记忆长 10
    
    # Q 取自解码器，K, V 取自编码器记忆
    out = cross_attn(q=dec_state, k=enc_memory, v=enc_memory)
    print("交叉注意力输出形状 (与解码器 Q 长度一致):", out.shape)
    assert out.shape == (2, 4, 64)

cross_attention_demo()

---
## 模块六：前馈神经网络 (Position-wise Feed-Forward Network, FFN)

### 【笔试考点】
- **经典结构**：
  $$\text{FFN}(x) = \max(0, x W_1 + b_1) W_2 + b_2$$
  中间隐层通常进行 **$4$ 倍升维**（$d_{ff} = 4 \times d_{model}$），现代多采用 GELU 激活函数替代 ReLU。

In [4]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff=256, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

ffn = PositionwiseFeedForward(64, 256)
ffn_out = ffn(x_in)
print("FFN 输出形状:", ffn_out.shape)
assert ffn_out.shape == (2, 6, 64)

NameError: name 'x_in' is not defined

---
## 模块七：Pre-LN vs. Post-LN 架构差异与残差连接

### 【笔试高频考点】
1. **Post-LN (原始 Transformer)**：$x_{l+1} = \text{LayerNorm}(x_l + \text{Sublayer}(x_l))$
   - 痛点：主干残差路径在深层被多次归一化缩放，梯度反传衰减严重，训练初期极其依赖 Warmup 且易崩溃。
2. **Pre-LN (主流现代大模型)**：$x_{l+1} = x_l + \text{Sublayer}(\text{LayerNorm}(x_l))$
   - 优势：残差主干 $x_l$ 直通无阻碍，梯度可以直接顺着恒等残差流回传到浅层，训练极为稳定！

In [ ]:
class PreLNSublayerConnection(nn.Module):
    """Pre-LN 残差连接模块: x + Sublayer(LayerNorm(x))"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

pre_ln = PreLNSublayerConnection(64)
norm_out = pre_ln(x_in, lambda z: mha(z, z, z))
print("Pre-LN 残差连接输出形状:", norm_out.shape)
assert norm_out.shape == (2, 6, 64)
print(">>> Pre-LN 恒等直通架构验证通过！")

---
## 模块八：标准 Transformer Encoder Block 与 Decoder Block 端到端组合

### 【笔试考点】
- **Encoder Block**：PreLN(Self-Attn) + PreLN(FFN)；
- **Decoder Block**：PreLN(Masked Self-Attn) + PreLN(Cross-Attn) + PreLN(FFN)。

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.sublayer1 = PreLNSublayerConnection(d_model)
        self.sublayer2 = PreLNSublayerConnection(d_model)

    def forward(self, x, mask=None):
        x = self.sublayer1(x, lambda z: self.self_attn(z, z, z, mask))
        x = self.sublayer2(x, self.ffn)
        return x

class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.sub1 = PreLNSublayerConnection(d_model)
        self.sub2 = PreLNSublayerConnection(d_model)
        self.sub3 = PreLNSublayerConnection(d_model)

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        x = self.sub1(x, lambda z: self.self_attn(z, z, z, tgt_mask))
        x = self.sub2(x, lambda z: self.cross_attn(z, memory, memory, src_mask))
        x = self.sub3(x, self.ffn)
        return x

# 验证 Block 堆叠
enc_block = TransformerEncoderBlock(d_model=64, num_heads=4, d_ff=128)
dec_block = TransformerDecoderBlock(d_model=64, num_heads=4, d_ff=128)

memory = enc_block(x_in)
dec_out = dec_block(x=torch.randn(2, 4, 64), memory=memory)
print("Encoder Memory 形状:", memory.shape)
print("Decoder Block 输出形状:", dec_out.shape)
assert memory.shape == (2, 6, 64) and dec_out.shape == (2, 4, 64)
print(">>> Transformer 核心 Block 组装全部通过！")

---
## 模块九：端到端极简自回归推理解码循环手撕

### 【笔试高频考题】
- **自回归推理机制**：
  从 `<BOS>` 开始，每轮将已生成的全部 Token 送入 Decoder，提取**最后一个时间步的 Logits**，通过 `argmax` 选出新词，追加到序列末尾，直至预测出 `<EOS>`。

In [ ]:
def autoregressive_generate_mock(encoder_memory, max_len=6, bos_idx=1, eos_idx=2):
    """
    自回归极简生成循环模拟
    """
    generated = torch.tensor([[bos_idx]]) # 初始化起始符 (1, 1)
    
    for step in range(max_len):
        # 模拟当前生成的 token 经过 decoder 得到 logits
        # 取最后一个时间步预测下一个 token
        next_token_id = (generated[0, -1].item() + 1) % 10 # 假定预测下一个词为上一词+1
        if next_token_id == eos_idx:
            generated = torch.cat([generated, torch.tensor([[eos_idx]])], dim=1)
            break
        generated = torch.cat([generated, torch.tensor([[next_token_id]])], dim=1)
        
    return generated.squeeze(0).tolist()

gen_seq = autoregressive_generate_mock(memory)
print("自回归循环生成 Token ID 序列:", gen_seq)
assert gen_seq[0] == 1
print(">>> 端到端自回归生成流程验证通过！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 除以根号 dk: 必须压制高维点积方差爆炸，保护 Softmax 远离零梯度饱和区。
2. MHA 维度变换: view 之前必加 .contiguous()，多头拆分顺序为 (B, H, L, d_k)。
3. 因果掩码下三角: tril 屏蔽未来时序，配合 masked_fill(mask==0, -1e9) 做 Softmax 截断。
4. Pre-LN 优于 Post-LN: Pre-LN 残差恒等直通梯度免衰减，无需深层 Warmup 也能收敛。
5. 交叉注意力来源: Q 来自解码器上一层，K, V 恒定来自编码器顶层 memory。
```